# SpaceRisk AI - Previsão Inteligente de Riscos Ambientais

Este notebook faz parte da entrega técnica de Machine Learning para a Global Solution 2026 (FIAP / Parceiro: Pacto Global da ONU).

**Equipe (Turma: 2TSCOR):**
- 559675 Luiz Felipe Duarte Silva
- 560446 Mateus Florencio Macedo
- 560577 Mickael Fabris dos Anjos
- 560096 Pedro Luiz dos Passos Aguiar
- 561034 Nicolas Samuel Crisostomo Neri

### Objetivo:
Desenvolver e comparar modelos supervisionados (**Decision Tree** e **Random Forest**) para classificar regiões em níveis de risco ambiental: **Baixo**, **Médio** ou **Alto**.

In [ ]:
# 1. Importação das Bibliotecas
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

# Estilo dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Bibliotecas importadas com sucesso!")

## 2. Carregamento do Dataset

In [ ]:
# Caminho do dataset
data_path = "../data/dados_ambientais.csv"
if not os.path.exists(data_path):
    # Caso execute a partir da raiz do projeto
    data_path = "data/dados_ambientais.csv"

df = pd.read_csv(data_path)
print(f"Dataset carregado! Formato: {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head()

## 3. Análise Exploratória dos Dados (EDA)

In [ ]:
# Informações gerais sobre as colunas
df.info()

In [ ]:
# Estatísticas descritivas
df.describe().T

In [ ]:
# Distribuição da Variável Alvo (Nivel_Risco)
plt.figure(figsize=(6, 4))
sns.countplot(x="Nivel_Risco", data=df, order=["Baixo", "Médio", "Alto"], palette="viridis")
plt.title("Distribuição do Nível de Risco Ambiental")
plt.xlabel("Nível de Risco")
plt.ylabel("Quantidade")
plt.show()

In [ ]:
# Correlação entre variáveis numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlação das Variáveis Numéricas")
plt.show()

In [ ]:
# Relação entre Temperatura, Umidade e o Risco Ambiental
plt.figure(figsize=(10, 6))
sns.scatterplot(x="Temperatura", y="Umidade", hue="Nivel_Risco", hue_order=["Baixo", "Médio", "Alto"], data=df, palette="RdYlGn_r", alpha=0.7)
plt.title("Relação Temperatura vs Umidade por Nível de Risco")
plt.show()

## 4. Pré-processamento e Tratamento de Dados

In [ ]:
# Verificando valores nulos
nulos = df.isnull().sum()
print("Valores Nulos por Coluna:")
print(nulos)

# Se houvesse nulos, poderíamos tratar com preenchimento (ex: df.fillna(df.mean()))
# Como nosso gerador não gera nulos, apenas mostramos que a verificação foi feita.

In [ ]:
# Codificação das variáveis categóricas (Label Encoding)
categorical_cols = ["Estado", "Bioma", "Mês", "Uso_Solo"]
encoders = {}

df_encoded = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col])
    encoders[col] = le

print("Variáveis categóricas codificadas com LabelEncoder!")

In [ ]:
# Mapeamento da variável alvo de texto para número para treinamento
target_map = {"Baixo": 0, "Médio": 1, "Alto": 2}
df_encoded["Nivel_Risco_Num"] = df_encoded["Nivel_Risco"].map(target_map)

# Divisão entre Variáveis Descritivas (X) e Alvo (y)
X = df_encoded.drop(columns=["Nivel_Risco", "Nivel_Risco_Num"])
y = df_encoded["Nivel_Risco_Num"]

print(f"X shape: {X.shape}, y shape: {y.shape}")

## 5. Divisão Treino/Teste e Padronização

In [ ]:
# Divisão de 80% para treino e 20% para teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Padronização das variáveis (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Treino: {X_train_scaled.shape[0]} amostras")
print(f"Teste: {X_test_scaled.shape[0]} amostras")

## 6. Treinamento dos Modelos

Vamos treinar dois algoritmos supervisionados:
1. **Decision Tree Classifier (Árvore de Decisão)**
2. **Random Forest Classifier (Floresta Aleatória)**

In [ ]:
# Modelo 1: Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train_scaled, y_train)

# Predições
y_pred_dt = dt_model.predict(X_test_scaled)
print("Modelo Decision Tree treinado com sucesso!")

In [ ]:
# Modelo 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predições
y_pred_rf = rf_model.predict(X_test_scaled)
print("Modelo Random Forest treinado com sucesso!")

## 7. Avaliação e Comparação dos Modelos

In [ ]:
# Função auxiliar para obter métricas detalhadas
def obter_metricas(y_true, y_pred, nome_modelo):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted")
    rec = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")
    
    return {
        "Modelo": nome_modelo,
        "Acurácia": acc,
        "Precisão": prec,
        "Recall": rec,
        "F1-Score": f1
    }

metricas_dt = obter_metricas(y_test, y_pred_dt, "Decision Tree")
metricas_rf = obter_metricas(y_test, y_pred_rf, "Random Forest")

df_metricas = pd.DataFrame([metricas_dt, metricas_rf])
df_metricas

In [ ]:
# Relatório de Classificação Completo
print("=== RELATÓRIO DE CLASSIFICAÇÃO: DECISION TREE ===")
print(classification_report(y_test, y_pred_dt, target_names=["Baixo", "Médio", "Alto"]))

print("=== RELATÓRIO DE CLASSIFICAÇÃO: RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf, target_names=["Baixo", "Médio", "Alto"]))

In [ ]:
# Plot de Matrizes de Confusão
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de Confusão - Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Blues", xticklabels=["Baixo", "Médio", "Alto"], yticklabels=["Baixo", "Médio", "Alto"], ax=ax[0])
ax[0].set_title("Matriz de Confusão - Decision Tree")
ax[0].set_ylabel("Valor Real")
ax[0].set_xlabel("Valor Previsto")

# Matriz de Confusão - Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", xticklabels=["Baixo", "Médio", "Alto"], yticklabels=["Baixo", "Médio", "Alto"], ax=ax[1])
ax[1].set_title("Matriz de Confusão - Random Forest")
ax[1].set_ylabel("Valor Real")
ax[1].set_xlabel("Valor Previsto")

plt.tight_layout()
plt.show()

## 8. Explicabilidade: Importância das Variáveis

A importância das variáveis indica quais fatores foram mais relevantes para que o modelo fizesse suas previsões de risco.

In [ ]:
# Importância das variáveis no Random Forest
importancias = rf_model.feature_importances_
nomes_features = X.columns

df_importancias = pd.DataFrame({
    "Feature": nomes_features,
    "Importancia": importancias
}).sort_values(by="Importancia", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x="Importancia", y="Feature", data=df_importancias, palette="mako")
plt.title("Importância das Variáveis para Previsão de Risco Ambiental (Random Forest)")
plt.xlabel("Importância Relativa")
plt.ylabel("Variável")
plt.show()

## 9. Exportação dos Modelos

Salvaremos o melhor modelo treinado (Random Forest) e os pré-processadores para que possam ser utilizados no nosso dashboard em tempo real (Streamlit).

In [ ]:
# Criar pasta de saída se não existir
os.makedirs("../modelos_salvos", exist_ok=True)
os.makedirs("modelos_salvos", exist_ok=True)

# Salvar artefatos
caminho_modelo = "modelos_salvos/spacerisk_rf_model.joblib"
caminho_scaler = "modelos_salvos/spacerisk_scaler.joblib"
caminho_encoders = "modelos_salvos/spacerisk_encoders.joblib"

joblib.dump(rf_model, caminho_modelo)
joblib.dump(scaler, caminho_scaler)
joblib.dump(encoders, caminho_encoders)

print("Modelos e pré-processadores salvos com sucesso na pasta 'modelos_salvos'!")